In [1]:
import numpy as np 
import pandas as pd 
import yfinance as yf 
from pathlib import Path 

ROOT = Path.cwd().parent
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"

print("Root:", ROOT)
print("Raw data:", DATA_RAW)
print("Processed data:", DATA_PROCESSED)

Root: c:\Users\ryanr\vrp_strategy
Raw data: c:\Users\ryanr\vrp_strategy\data\raw
Processed data: c:\Users\ryanr\vrp_strategy\data\processed


In [2]:
# Download from start and consolidate for data gaps
START_DATE = "1990-01-01"
def fetch_and_save_raw_data(ticker, name):
    print(f"Fetching {ticker}...")
    df = yf.download(ticker, START_DATE, auto_adjust=True, progress=False)
    df = df[["Close"]].rename(columns={"Close": name})
    path = DATA_RAW / f"{name}.csv"
    df.to_csv(path)
    print(f"  {name}: {df.index[0].date()} -> {df.index[-1].date()} ({len(df)} rows)")
    return df

spx  = fetch_and_save_raw_data("^GSPC", "spx")
vix  = fetch_and_save_raw_data("^VIX",  "vix")
vix3m = fetch_and_save_raw_data("^VIX3M", "vix3m")


Fetching ^GSPC...
  spx: 1990-01-02 -> 2026-05-22 (9165 rows)
Fetching ^VIX...
  vix: 1990-01-02 -> 2026-05-25 (9166 rows)
Fetching ^VIX3M...
  vix3m: 2006-07-17 -> 2026-05-22 (4995 rows)


In [3]:
#VIX3M has least data, inner join on spx and vix. 
data = spx.join(vix, how="inner").join(vix3m, how="inner")
data.index = pd.to_datetime(data.index)
data.index.name = "date"
data.columns = ["spx", "vix", "vix3m"]
data = data.sort_index()
print(f"Total Trading Days: {len(data)}")
print(data.head())

#Save to processed data in master.csv
path = DATA_PROCESSED / "master.csv"
data.to_csv(path)
print(f"\nSaved -> {path.name}")

Total Trading Days: 4995
                    spx        vix      vix3m
date                                         
2006-07-17  1234.489990  18.639999  18.049999
2006-07-18  1236.859985  17.740000  17.219999
2006-07-19  1259.810059  15.550000  15.600000
2006-07-20  1249.130005  16.209999  16.309999
2006-07-21  1240.290039  17.400000  17.040001

Saved -> master.csv
